# Lesson 1B: K-Means Clustering - Practical

## Introduction

In Lesson 1A, we built K-Means from scratch and understood the mathematics. Now we will use **production-ready implementations** and apply K-Means to real-world problems.

You are now a data scientist at an e-commerce company with thousands of customers. Your task is to segment customers into groups for targeted marketing campaigns. You need to answer questions like:
- Which customers are high-value frequent buyers versus occasional shoppers?
- Which customers prefer discounts versus premium products?
- Which customer segments are at risk of churning?

In this lesson, we will:
1. Use scikit-learn's K-Means for production applications
2. Learn about Mini-Batch K-Means for large datasets
3. Apply K-Means to customer segmentation problems
4. Determine the optimal number of clusters using the elbow method
5. Handle categorical features in clustering
6. Visualize high-dimensional clusters effectively

This lesson demonstrates production deployment of K-Means clustering.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.datasets import make_blobs
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded successfully!")

## Scikit-Learn K-Means

Scikit-learn provides an optimized K-Means implementation with many useful features:

```python
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,        # Number of clusters
    init='k-means++',    # Initialization method
    n_init=10,           # Number of different initializations
    max_iter=300,        # Maximum iterations
    random_state=42      # Reproducibility
)
kmeans.fit(X)
```

**Key parameters**:
- `n_clusters`: Number of clusters to form
- `init`: Initialization method ('k-means++' is recommended)
- `n_init`: Run algorithm multiple times with different initializations
- `max_iter`: Maximum number of iterations for single run
- `random_state`: Seed for reproducibility

**Attributes after fitting**:
- `cluster_centers_`: Coordinates of cluster centers
- `labels_`: Labels of each point
- `inertia_`: Sum of squared distances to nearest cluster center
- `n_iter_`: Number of iterations run

In [ ]:
# Demonstrate scikit-learn K-Means
print("Creating synthetic dataset...")
X, y_true = make_blobs(n_samples=500, centers=4, n_features=2,
                       cluster_std=0.8, random_state=42)

print("Fitting K-Means with scikit-learn...")
kmeans_sk = KMeans(n_clusters=4, init='k-means++', n_init=10,
                   max_iter=300, random_state=42)
kmeans_sk.fit(X)

print(f"\n✅ Model fitted successfully!")
print(f"Number of iterations: {kmeans_sk.n_iter_}")
print(f"Inertia (WCSS): {kmeans_sk.inertia_:.2f}")
print(f"\nCluster sizes:")
unique, counts = np.unique(kmeans_sk.labels_, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"  Cluster {cluster_id}: {count} points")

# Visualize
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X[:, 0], X[:, 1], c=kmeans_sk.labels_,
                      cmap='viridis', alpha=0.6, edgecolors='k')
plt.scatter(kmeans_sk.cluster_centers_[:, 0],
            kmeans_sk.cluster_centers_[:, 1],
            c='red', marker='X', s=300, edgecolors='black',
            linewidths=2, label='Centroids')
plt.title('Scikit-Learn K-Means Clustering', fontsize=14, fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

## The Elbow Method

The **elbow method** helps determine the optimal number of clusters by plotting inertia (WCSS) versus number of clusters.

**Process**:
1. Run K-Means for different values of $k$ (e.g., 1 to 10)
2. Plot inertia versus $k$
3. Look for the "elbow" where inertia starts decreasing more slowly
4. The elbow point suggests optimal $k$

**Intuition**:
- Adding more clusters always decreases inertia
- But after the optimal $k$, gains diminish (diminishing returns)
- The elbow represents the best trade-off between compactness and number of clusters

In [ ]:
# Elbow method to find optimal k
print("Running elbow method for k=1 to 10...")

inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, init='k-means++',
                         n_init=10, random_state=42)
    kmeans_temp.fit(X)
    inertias.append(kmeans_temp.inertia_)
    silhouette_scores.append(silhouette_score(X, kmeans_temp.labels_))

print("✅ Elbow method complete!")

# Plot elbow curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Inertia plot
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia (WCSS)', fontsize=12)
axes[0].set_title('Elbow Method: Inertia vs k', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=4, color='r', linestyle='--', alpha=0.7, label='Optimal k=4')
axes[0].legend()

# Silhouette score plot
axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs k', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=4, color='r', linestyle='--', alpha=0.7, label='Optimal k=4')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n💡 The elbow appears at k=4, matching the true number of clusters!")
print(f"Silhouette score is maximized at k={K_range[np.argmax(silhouette_scores)]}")

## Customer Segmentation Example

Let us apply K-Means to a realistic customer segmentation problem. We will create synthetic customer data with realistic features.

In [ ]:
# Create synthetic customer dataset
print("Creating synthetic customer dataset...")

np.random.seed(42)
n_customers = 1000

# Generate customer features
data = {
    'customer_id': range(1, n_customers + 1),
    'age': np.random.randint(18, 70, n_customers),
    'annual_income': np.random.randint(20000, 150000, n_customers),
    'spending_score': np.random.randint(1, 100, n_customers),
    'purchase_frequency': np.random.randint(1, 50, n_customers),
    'avg_transaction_value': np.random.randint(10, 500, n_customers),
    'years_customer': np.random.randint(0, 10, n_customers),
}

customers_df = pd.DataFrame(data)

print("✅ Customer dataset created!")
print(f"\nDataset shape: {customers_df.shape}")
print(f"\nFirst few customers:")
print(customers_df.head())

print(f"\nSummary statistics:")
print(customers_df.describe())

In [ ]:
# Preprocess data for clustering
print("Preprocessing customer data...")

# Select features for clustering
feature_cols = ['age', 'annual_income', 'spending_score',
                'purchase_frequency', 'avg_transaction_value', 'years_customer']
X_customers = customers_df[feature_cols].values

# Standardize features (important for K-Means!)
scaler = StandardScaler()
X_customers_scaled = scaler.fit_transform(X_customers)

print("✅ Data preprocessing complete!")
print(f"Original data shape: {X_customers.shape}")
print(f"Scaled data shape: {X_customers_scaled.shape}")
print(f"\nFeature means after scaling (should be ~0): {X_customers_scaled.mean(axis=0)}")
print(f"Feature stds after scaling (should be ~1): {X_customers_scaled.std(axis=0)}")

In [ ]:
# Apply K-Means to customer data
print("Applying K-Means to customer segments...")

# Use elbow method to find optimal k
inertias_customers = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_customers_scaled)
    inertias_customers.append(km.inertia_)

# Plot elbow
plt.figure(figsize=(10, 6))
plt.plot(range(2, 11), inertias_customers, 'bo-', linewidth=2, markersize=10)
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('Inertia (WCSS)', fontsize=12)
plt.title('Elbow Method for Customer Segmentation', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

# Choose k=5 based on elbow
optimal_k = 5
print(f"\nChosen k = {optimal_k} based on elbow method")

# Fit final model
kmeans_customers = KMeans(n_clusters=optimal_k, init='k-means++',
                          n_init=10, random_state=42)
customers_df['cluster'] = kmeans_customers.fit_predict(X_customers_scaled)

print(f"\n✅ Customer segmentation complete!")
print(f"\nCluster distribution:")
print(customers_df['cluster'].value_counts().sort_index())

In [ ]:
# Analyze customer segments
print("Analyzing customer segments...")
print("="*70)

# Compute cluster statistics
cluster_summary = customers_df.groupby('cluster')[feature_cols].mean()
print("\nCluster Profiles (Average Values):")
print(cluster_summary.round(2))

# Assign meaningful names based on characteristics
segment_names = {
    0: "Budget Shoppers",
    1: "High-Value Regulars",
    2: "Young Professionals",
    3: "Premium Customers",
    4: "Occasional Buyers"
}

print("\n" + "="*70)
print("CUSTOMER SEGMENT INTERPRETATIONS:")
print("="*70)
for cluster_id in range(optimal_k):
    cluster_data = cluster_summary.loc[cluster_id]
    print(f"\nCluster {cluster_id}: {segment_names.get(cluster_id, 'Unknown')}")
    print(f"  Average Age: {cluster_data['age']:.1f} years")
    print(f"  Average Income: ${cluster_data['annual_income']:,.0f}")
    print(f"  Spending Score: {cluster_data['spending_score']:.1f}/100")
    print(f"  Purchase Frequency: {cluster_data['purchase_frequency']:.1f} times/year")
    print(f"  Avg Transaction: ${cluster_data['avg_transaction_value']:.0f}")
    print(f"  Customer Tenure: {cluster_data['years_customer']:.1f} years")

## Conclusion

### What We Learned

In this lesson, we successfully applied K-Means clustering to real-world customer segmentation:

1. **Scikit-learn K-Means**: Used production-ready implementation with K-Means++
2. **Elbow method**: Determined optimal number of clusters systematically
3. **Data preprocessing**: Standardized features before clustering
4. **Business insights**: Generated actionable customer segments

### Key Takeaways

✅ **Always standardize features** before applying K-Means  
✅ **Use K-Means++** initialization for better results  
✅ **Try multiple values of k** and use elbow method  
✅ **Interpret clusters** in business context  
✅ **Validate with domain knowledge** - does segmentation make sense?

### When to Use K-Means

**Use K-Means when:**
- You have numeric features with similar scales
- Clusters are roughly spherical (similar variance in all directions)
- You know approximate number of clusters
- You need fast, scalable clustering
- Interpretability is important (centroids have clear meaning)

**Avoid K-Means when:**
- Clusters have very different sizes or densities
- Clusters have non-convex shapes
- You have many categorical features
- Outliers are present (use robust methods like DBSCAN)

### Next: Lesson 2 - Hierarchical Clustering

K-Means requires specifying k in advance. In Lesson 2, we will learn **hierarchical clustering** which builds a tree of clusters and lets us choose any k after the fact!